# 05 — Text Embeddings + Qdrant Similarity Search

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

In [ ]:
CLEAN_CSV       = '../data/movies_cleaned.csv'
MODEL_NAME      = 'all-MiniLM-L6-v2'
EMBEDDING_DIM   = 384
COLLECTION_NAME = 'movies_overview'
QDRANT_HOST     = 'localhost'
QDRANT_PORT     = 6333

print(f'Model:      {MODEL_NAME}')
print(f'Dimension:  {EMBEDDING_DIM}')
print(f'Collection: {COLLECTION_NAME}')
print(f'Qdrant:     {QDRANT_HOST}:{QDRANT_PORT}')

## 2. Load Movie Data

In [ ]:
movies = pd.read_csv(CLEAN_CSV)
print(f'Loaded {len(movies):,} movies')
movies[['title', 'release_year', 'overview']].head(3)

In [ ]:
movies = movies.dropna(subset=['overview']).reset_index(drop=True)
movies['overview'] = movies['overview'].astype(str)
movies['overview_length'] = movies['overview'].str.len()

print(f'Movies after dropping null overviews: {len(movies):,}')
print(f'\nOverview length statistics:')
print(movies['overview_length'].describe())

## 3. Generate Embeddings

In [ ]:
print(f'Loading model: {MODEL_NAME} ...')
model = SentenceTransformer(MODEL_NAME)
print(f'Model loaded. Output dimension: {model.get_sentence_embedding_dimension()}')

In [ ]:
texts = movies['overview'].tolist()

print(f'Encoding {len(texts):,} overviews...')
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print(f'\nEmbeddings shape: {embeddings.shape}')
print(f'Sample (first 5 dims of first vector): {embeddings[0][:5]}')

## 4. Connect to Qdrant and Create Collection

In [ ]:
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)
print(f'Connected to Qdrant at {QDRANT_HOST}:{QDRANT_PORT}')

collections = client.get_collections().collections
print(f'Existing collections: {[c.name for c in collections]}')

In [ ]:
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)
    print(f'Deleted existing collection: {COLLECTION_NAME}')

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=EMBEDDING_DIM,
        distance=Distance.COSINE,
    ),
)
print(f'Created collection: {COLLECTION_NAME}')

## 5. Upsert Points with Payload

In [ ]:
points = []
for i, row in movies.iterrows():
    payload = {
        'node_id':       str(row['title']),
        'raw_text':      str(row['overview']),
        'release_year':  int(row['release_year']) if pd.notna(row['release_year']) else None,
        'imdb_rating':   float(row['imdb_rating']) if pd.notna(row['imdb_rating']) else None,
        'revenue':       float(row['revenue']) if pd.notna(row['revenue']) else None,
        'director':      str(row['director']) if pd.notna(row['director']) else None,
        'actor':         str(row['actor']) if pd.notna(row['actor']) else None,
        'genres_list':   str(row['genres_list']) if pd.notna(row['genres_list']) else None,
    }
    points.append(PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload=payload,
    ))

print(f'Built {len(points):,} points ready for upsert')

In [ ]:
BATCH_SIZE = 256
for start in range(0, len(points), BATCH_SIZE):
    batch = points[start:start + BATCH_SIZE]
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch,
    )

info = client.get_collection(COLLECTION_NAME)
print(f'Upsert complete.')
print(f'Collection now contains: {info.points_count:,} points')

## 6. Similarity Query — Find Movies by Text

In [ ]:
query_text = 'A space adventure with aliens and intergalactic battles'

print(f'QUERY: "{query_text}"')
print('=' * 80)

query_vector = model.encode(query_text).tolist()

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=5,
    with_payload=True,
)

for rank, hit in enumerate(results.points, 1):
    p = hit.payload
    print(f"\n#{rank}  score = {hit.score:.4f}")
    print(f"     Title:    {p['node_id']}  ({p['release_year']})")
    print(f"     Director: {p['director']}")
    print(f"     Genres:   {p['genres_list']}")
    print(f"     Overview: {p['raw_text'][:160]}...")

In [ ]:
query_text = 'A heartwarming story about family bonds and overcoming challenges'

print(f'QUERY: "{query_text}"')
print('=' * 80)

query_vector = model.encode(query_text).tolist()

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=5,
    with_payload=True,
)

for rank, hit in enumerate(results.points, 1):
    p = hit.payload
    print(f"\n#{rank}  score = {hit.score:.4f}")
    print(f"     Title:    {p['node_id']}  ({p['release_year']})")
    print(f"     Director: {p['director']}")
    print(f"     Genres:   {p['genres_list']}")
    print(f"     Overview: {p['raw_text'][:160]}...")

## 7. Movie-to-Movie Similarity

In [ ]:
seed_title = 'Inception'

seed_idx = movies[movies['title'] == seed_title].index
if len(seed_idx) == 0:
    print(f'Movie "{seed_title}" not found in dataset.')
else:
    seed_idx = seed_idx[0]
    seed_overview = movies.loc[seed_idx, 'overview']
    seed_vector   = embeddings[seed_idx].tolist()

    print(f'SEED MOVIE: {seed_title}')
    print(f'OVERVIEW:   {seed_overview[:200]}...')
    print('=' * 80)
    print('TOP 5 MOST SIMILAR MOVIES:')

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=seed_vector,
        limit=6,
        with_payload=True,
    )

    rank = 0
    for hit in results.points:
        if hit.payload['node_id'] == seed_title:
            continue
        rank += 1
        p = hit.payload
        print(f"\n#{rank}  score = {hit.score:.4f}")
        print(f"     Title:    {p['node_id']}  ({p['release_year']})")
        print(f"     Director: {p['director']}")
        print(f"     Genres:   {p['genres_list']}")
        print(f"     Overview: {p['raw_text'][:160]}...")
        if rank >= 5:
            break

## 8. Filtered Similarity Query

In [ ]:
from qdrant_client.models import Filter, FieldCondition, Range

query_text = 'time travel and parallel universes with mind-bending plot'
query_vector = model.encode(query_text).tolist()

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    query_filter=Filter(
        must=[
            FieldCondition(key='release_year', range=Range(gte=2010)),
            FieldCondition(key='imdb_rating',  range=Range(gte=7.0)),
        ]
    ),
    limit=5,
    with_payload=True,
)

print(f'QUERY: "{query_text}"')
print('FILTERS: release_year >= 2010, imdb_rating >= 7.0')
print('=' * 80)

for rank, hit in enumerate(results.points, 1):
    p = hit.payload
    print(f"\n#{rank}  score = {hit.score:.4f}")
    print(f"     Title:    {p['node_id']}  ({p['release_year']})  rating={p['imdb_rating']}")
    print(f"     Director: {p['director']}")
    print(f"     Genres:   {p['genres_list']}")
    print(f"     Overview: {p['raw_text'][:160]}...")